# 10 — Analysis: confusion deltas, Grad-CAM, failure cases (Phase 11)

Phase 10 produced the headline numbers. **Phase 11 explains them.** Three blocks:

1. **Confusion matrix delta** — baseline vs augmented, side-by-side, plus a delta matrix showing where errors shifted. This is what tells us *which* misclassifications the synthetic data fixed and which new ones it introduced (if any).
2. **Grad-CAM** on the augmented model — for each rare class, show what regions the model attends to when it makes a correct prediction. This is the "did it learn the lesion or the background" question, which is the standard worry for medical-image classifiers and is what the discussion section of the report needs.
3. **Failure-case montage** — the worst confusions on the test set (especially MEL → NV, which is clinically dangerous because melanoma is the malignant class). Real images, predicted vs true.

**Reference:** Grad-CAM (Selvaraju et al. 2017): hook the last conv block (`layer4` for ResNet50), capture gradients, weight feature maps by their gradient averages → heatmap.

**Outputs:**
- `results/plots/confusion_baseline_vs_augmented.png` (side-by-side + delta)
- `results/plots/gradcam_correct_rare_classes.png` (one row per rare class)
- `results/plots/failure_cases.png` (top confusions, real images)
- `results/logs/confusion_delta.csv` (numeric delta matrix for the appendix)

## 1. Setup, paths, load both models

In [ ]:
import sys
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import confusion_matrix

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import ISICDataset, CLASS_NAMES                # noqa: E402
from src.models.classifier import build_resnet50_classifier     # noqa: E402

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

RESULTS    = PROJECT_ROOT / 'results'
SPLITS_CSV = PROJECT_ROOT / 'data' / 'processed' / 'splits.csv'
IMAGES_DIR = PROJECT_ROOT / 'data' / 'raw' / 'ISIC2018_Task3_Training_Input'
BASELINE_CKPT  = RESULTS / 'checkpoints' / 'baseline_resnet50_best.pt'
AUGMENTED_CKPT = RESULTS / 'checkpoints' / 'augmented_resnet50_best.pt'

(RESULTS / 'logs').mkdir(parents=True, exist_ok=True)
(RESULTS / 'plots').mkdir(parents=True, exist_ok=True)

for p in [BASELINE_CKPT, AUGMENTED_CKPT]:
    assert p.exists(), f'Missing checkpoint: {p}'

N_CLASSES = len(CLASS_NAMES)
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
test_ds = ISICDataset(SPLITS_CSV, IMAGES_DIR, split='test', transform=eval_tf)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
print(f'test: {len(test_ds)} samples ({len(test_loader)} batches)')

# Load both models
def load_classifier(ckpt_path):
    model = build_resnet50_classifier(n_classes=N_CLASSES, freeze_backbone=True).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    state = ckpt.get('model_state_dict', ckpt.get('state_dict'))
    model.load_state_dict(state)
    model.eval()
    return model

baseline_model  = load_classifier(BASELINE_CKPT)
augmented_model = load_classifier(AUGMENTED_CKPT)
print('Both models loaded.')

## 2. Run both models over the test set (single pass each)

We also keep the test images and their paths around because Grad-CAM and the failure montage need them.

In [ ]:
def predict_all(model):
    preds, labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE, non_blocking=True)
            preds.extend(model(x).argmax(1).cpu().tolist())
            labels.extend(y.tolist())
    return np.array(preds), np.array(labels)

base_preds, test_labels = predict_all(baseline_model)
aug_preds,  _          = predict_all(augmented_model)
print('Predictions collected.')
print(f'  baseline  agrees with augmented on {(base_preds == aug_preds).mean()*100:.1f}% of test images')

## 3. Side-by-side + delta confusion matrices

Three panels: baseline | augmented | (augmented − baseline). The delta panel uses a diverging colormap — green = augmented does better here (more samples on this row/col combo where it should be, fewer where it shouldn't); red = augmented does worse.

In [ ]:
labels_arr = list(range(N_CLASSES))
cm_base = confusion_matrix(test_labels, base_preds, labels=labels_arr)
cm_aug  = confusion_matrix(test_labels, aug_preds,  labels=labels_arr)

# Row-normalised so diagonals = per-class recall
cm_base_n = cm_base.astype(float) / cm_base.sum(axis=1, keepdims=True).clip(min=1)
cm_aug_n  = cm_aug.astype(float)  / cm_aug.sum(axis=1, keepdims=True).clip(min=1)
cm_delta  = cm_aug_n - cm_base_n   # positive = augmented adds mass here

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

def draw_cm(ax, mat, title, cmap, vmin, vmax, fmt='{:.2f}'):
    im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title(title)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            v = mat[i, j]
            # contrast-aware text colour
            if cmap == 'Blues':
                color = 'white' if v > 0.5 else 'black'
            else:
                color = 'black' if abs(v) < 0.25 else 'white'
            ax.text(j, i, fmt.format(v), ha='center', va='center', color=color, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

draw_cm(axes[0], cm_base_n, 'Baseline (Phase 5)',  'Blues', 0, 1)
draw_cm(axes[1], cm_aug_n,  'Augmented (Phase 10)', 'Blues', 0, 1)
draw_cm(axes[2], cm_delta,  'Delta (augmented − baseline)',
        'RdYlGn', vmin=-0.5, vmax=0.5, fmt='{:+.2f}')

plt.tight_layout()
out = RESULTS / 'plots' / 'confusion_baseline_vs_augmented.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
print(f'saved {out}')
plt.show()

# Also dump the delta as CSV for the appendix
delta_df = pd.DataFrame(cm_delta, index=CLASS_NAMES, columns=CLASS_NAMES)
delta_df.to_csv(RESULTS / 'logs' / 'confusion_delta.csv')
print(f'saved {RESULTS / "logs" / "confusion_delta.csv"}')
print('\nWhere did the augmented model gain/lose mass? (diagonals = recall gains)')
for c in range(N_CLASSES):
    print(f'  {CLASS_NAMES[c]:<8} diag delta = {cm_delta[c, c]:+.3f}')

## 4. Grad-CAM implementation

Standard Grad-CAM (Selvaraju et al. 2017):
1. Forward pass, capture the activation map at the last conv block (`model.layer4`).
2. Backward pass from the target class logit, capture the gradients at that same point.
3. Weight each channel of the activation by the global-average gradient → weighted sum across channels → ReLU → resize to image size.

We do all of this manually with forward/backward hooks (no extra dependency).

In [ ]:
class GradCAM:
    """Grad-CAM for a chosen conv layer in a model.

    Default target: ResNet50's `layer4` (the final conv block, 7x7 spatial res
    at 224 input). Choose later layers for high-level semantic localisation.
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._fwd_h = target_layer.register_forward_hook(self._save_act)
        self._bwd_h = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):
        self.activations = out.detach()

    def _save_grad(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def close(self):
        self._fwd_h.remove(); self._bwd_h.remove()

    def __call__(self, x, class_idx):
        """Run Grad-CAM for input x (1, 3, H, W) and target class index.
        Returns a (H, W) heatmap in [0, 1].
        """
        self.model.zero_grad()
        x = x.to(DEVICE).requires_grad_(False)
        logits = self.model(x)                                # (1, n_classes)
        target = logits[0, class_idx]
        target.backward()

        # weight = global average of gradients per channel
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)     # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True) # (1, 1, h, w)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE),
                            mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        # min-max normalise to [0, 1]
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = np.zeros_like(cam)
        return cam

# Quick test on one image to verify the plumbing
gcam = GradCAM(augmented_model, augmented_model.layer4)
x0, y0 = test_ds[0]
heat = gcam(x0.unsqueeze(0), int(y0))
print(f'Grad-CAM test: heatmap shape={heat.shape}, min={heat.min():.2f}, max={heat.max():.2f}  (expected (224,224), 0..1)')
gcam.close()

## 5. Grad-CAM for correct rare-class predictions

For each rare class (BCC, AKIEC, VASC, DF), pick up to 3 test images that the **augmented** model classified correctly, and show the Grad-CAM heatmap overlaid on the original image. This is what you put in the report to argue "the model learned to look at the lesion, not the surroundings."

If the heatmaps are scattered or on background skin, that's a negative finding worth reporting honestly: the model got lucky on these classes for the wrong reasons.

In [ ]:
RARE_CLASSES = [2, 3, 5, 6]    # BCC, AKIEC, DF, VASC
N_PER_CLASS  = 3

# Index test_ds by (true label, predicted correctly by augmented model)
correct_mask = (aug_preds == test_labels)
indices_by_class = {c: np.where((test_labels == c) & correct_mask)[0]
                    for c in RARE_CLASSES}

def denormalise(x):
    """Invert ImageNet normalisation for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

def overlay(img_rgb, heat, alpha=0.45):
    """Blend a heatmap onto an RGB image. img_rgb in [0,1], heat in [0,1]."""
    cmap = plt.get_cmap('jet')
    heat_rgb = cmap(heat)[..., :3]   # drop alpha
    return (1 - alpha) * img_rgb + alpha * heat_rgb

gcam = GradCAM(augmented_model, augmented_model.layer4)

fig, axes = plt.subplots(len(RARE_CLASSES), N_PER_CLASS * 2,
                         figsize=(2.2 * N_PER_CLASS * 2, 2.4 * len(RARE_CLASSES)))

for r, c in enumerate(RARE_CLASSES):
    idxs = indices_by_class[c][:N_PER_CLASS]
    for j in range(N_PER_CLASS):
        ax_img  = axes[r, j * 2]
        ax_heat = axes[r, j * 2 + 1]

        if j >= len(idxs):
            ax_img.axis('off'); ax_heat.axis('off')
            ax_img.text(0.5, 0.5, 'no correct\nprediction', ha='center', va='center', fontsize=9)
            continue

        idx = int(idxs[j])
        x, y = test_ds[idx]
        rgb = denormalise(x)
        heat = gcam(x.unsqueeze(0), int(y))

        ax_img.imshow(rgb);  ax_img.axis('off')
        ax_heat.imshow(overlay(rgb, heat)); ax_heat.axis('off')
        if j == 0:
            ax_img.set_ylabel(CLASS_NAMES[c], fontsize=11, rotation=0, labelpad=30, va='center')
        if r == 0:
            ax_img.set_title('original', fontsize=10)
            ax_heat.set_title('Grad-CAM', fontsize=10)

gcam.close()
plt.suptitle('Grad-CAM on correct rare-class predictions (augmented model)', y=1.01)
plt.tight_layout()
out = RESULTS / 'plots' / 'gradcam_correct_rare_classes.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
print(f'saved {out}')
plt.show()

## 6. Failure-case montage

Pick the worst confusions on the augmented model. We focus on:
- **MEL → NV**: clinically dangerous false negative (missed melanoma)
- **NV → MEL**: false alarm but cheap (extra biopsy)
- The two largest off-diagonal cells of the augmented confusion matrix overall

Show real images for each, with predicted vs true labels. This is what makes the report's "limitations" section concrete instead of hand-wavy.

In [ ]:
# Pick interesting confusion cells
MEL, NV = 0, 1

# Top 2 largest off-diagonal cells of the augmented confusion matrix
off_diag = cm_aug.copy().astype(float)
np.fill_diagonal(off_diag, 0)
flat_top = np.argsort(off_diag, axis=None)[::-1][:2]
top_pairs = [tuple(divmod(i, N_CLASSES)) for i in flat_top]   # (true, pred)

interesting = [(MEL, NV), (NV, MEL)] + [p for p in top_pairs if p not in [(MEL, NV), (NV, MEL)]]
interesting = interesting[:4]                                  # cap at 4 pairs

N_PER_PAIR = 4
fig, axes = plt.subplots(len(interesting), N_PER_PAIR,
                         figsize=(2.2 * N_PER_PAIR, 2.4 * len(interesting)))
if len(interesting) == 1:   # matplotlib weirdness when only 1 row
    axes = axes.reshape(1, -1)

for r, (true_c, pred_c) in enumerate(interesting):
    mask = (test_labels == true_c) & (aug_preds == pred_c)
    idxs = np.where(mask)[0][:N_PER_PAIR]

    for j in range(N_PER_PAIR):
        ax = axes[r, j]
        if j >= len(idxs):
            ax.axis('off')
            if j == 0:
                ax.text(0.5, 0.5, 'none', ha='center', va='center', fontsize=10)
            continue
        idx = int(idxs[j])
        x, _ = test_ds[idx]
        ax.imshow(denormalise(x))
        ax.axis('off')
        if j == 0:
            ax.set_ylabel(f'true {CLASS_NAMES[true_c]}\npred {CLASS_NAMES[pred_c]}\n(n={int(mask.sum())})',
                          fontsize=10, rotation=0, labelpad=50, va='center')

plt.suptitle('Failure cases — augmented model', y=1.01)
plt.tight_layout()
out = RESULTS / 'plots' / 'failure_cases.png'
plt.savefig(out, dpi=130, bbox_inches='tight')
print(f'saved {out}')
plt.show()

print('\nMost common confusions in the augmented model:')
for (t, p) in interesting:
    n = int(((test_labels == t) & (aug_preds == p)).sum())
    print(f'  true={CLASS_NAMES[t]:<6} pred={CLASS_NAMES[p]:<6}  n={n}')

## 7. Reading these results for the report

**Confusion delta:** the diagonal deltas are per-class recall changes already (positive = augmented improved this class's recall). Off-diagonal positive cells = augmented model now confuses these two classes more often; off-diagonal negative cells = it confused them less. Most interesting cell for the discussion: did the (true=MEL, pred=NV) cell shrink? That's a missed-melanoma rate.

**Grad-CAM:** the eyeball question is *do the hot zones overlap with the lesion?* If yes → the model learned the right thing. If no → it's using surrounding skin colour, ruler markers, or vignetting — common ISIC-data shortcuts. Mention this honestly either way.

**Failure cases:** for the MEL → NV row, are these visually atypical melanomas (small, light, no irregular border)? If so, that's a defensible limitation. If they look obviously like melanoma to a human eye, the model is failing for non-obvious reasons and that goes in the limitations section.

**Phase 12 next:** polish the headline figures (consistent fonts/colors/sizes), export a single PDF-friendly version of the per-class recall chart for the report. No new analysis after this — just typography.